In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Domains and EDAM categories
domains = [
    "Metagenomic",
    "Neuroimaging",
    "Phylogeny",
    "Single_Cell",
    "Systems_biology",
    "Genetic_variant",
    "Microscopy"
]
edam_categories = ["data", "formats", "operations", "topics"]

In [3]:
# Function to clean column names
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()              
        .str.replace(r"\s+", " ", regex=True)  
    )
    return df

In [4]:
processed_dfs = []

# Loop through files
for domain in domains:
    for category in edam_categories:
        file_path = f"{domain}/{domain}_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_{category}.tsv"

        if os.path.exists(file_path):
            df = pd.read_csv(file_path, sep="\t")
            df = clean_columns(df)

            # Supprimer les lignes complètement vides
            df = df.dropna(how="all")

            # Supprimer les lignes sans URI EDAM
            df = df.dropna(subset=["EDAM term URI"])
            df = df[df["EDAM term URI"].astype(str).str.strip() != ""]

            # Ajouter les métadonnées
            df["domain"] = domain
            df["edam_category"] = category

            processed_dfs.append(df)
            print(f"Processed {file_path}: {len(df)} rows kept.")

        else:
            print(f"Warning: File not found - {file_path}")

Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_data.tsv: 24 rows kept.
Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_formats.tsv: 10 rows kept.
Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_operations.tsv: 25 rows kept.
Processed Metagenomic/Metagenomic_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_topics.tsv: 15 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_data.tsv: 13 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_formats.tsv: 22 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_operations.tsv: 10 rows kept.
Processed Neuroimaging/Neuroimaging_gold_standard_consensus_expert_LLM_mapping_terms_EDAM_topics.tsv: 10 rows kept.
Processed Phylogeny/Phylogeny_gold_standard_consensus_expert_LLM_mapping_t

In [5]:
# Combine into master DataFrame
if processed_dfs:
    master_df = pd.concat(processed_dfs, ignore_index=True)

    # Une ligne par outil
    master_df["tools"] = master_df["tools"].str.split(r"\s*,\s*")
    master_df = master_df.explode("tools").reset_index(drop=True)

    # Nettoyer les noms des outils
    master_df["tools"] = master_df["tools"].str.strip()

    # Trier pour faciliter la lecture
    master_df = master_df.sort_values(
        by=["tools", "domain", "edam_category", "annotation"],
        ignore_index=True
    )

    display(master_df.head())
    print("\nColumns:")
    print(master_df.columns.tolist())

else:
    print("No data processed.")

,tools,annotation,candidate EDAM term label,Validation expert,Commentaire expert 1,Commentaire expert 2,hasExactmatch,hasBroader,hasNarrower,EDAM term URI,Comment,domain,edam_category,Commentaire expert 3
0,AFNI,anatomical mri,MRI image,Yes,NaN,NaN,No,Yes,No,http://edamontology.org/data_3442,NaN,Neuroimaging,data,NaN
1,AFNI,fmri,MRI image,Yes,"functional MRI, Functional magnetic resonance ...",NaN,No,Yes,No,http://edamontology.org/data_3442,NaN,Neuroimaging,data,NaN
2,AFNI,parametric modulation maps,Statistical estimate score,Yes,Statistical estimations,NaN,No,Yes,No,http://edamontology.org/data_0951,NaN,Neuroimaging,data,NaN
3,AFNI,nifti,Image format,Yes,NaN,NaN,No,No,No,http://edamontology.org/format_3549,NaN,Neuroimaging,formats,NaN
4,AFNI,group statistical analysis,Statistical calculation,Yes,NaN,NaN,No,No,No,http://edamontology.org/operation_2238,NaN,Neuroimaging,operations,NaN



Columns:
['tools', 'annotation', 'candidate EDAM term label', 'Validation expert', 'Commentaire expert 1', 'Commentaire expert 2', 'hasExactmatch', 'hasBroader', 'hasNarrower', 'EDAM term URI', 'Comment', 'domain', 'edam_category', 'Commentaire expert 3']


In [ ]:
master_df.to_csv(
    "All_Domains_GT_consensus_expert_LLM_mapping_terms_EDAM.tsv",
    sep="\t",
    index=False
)


In [30]:
validated_df = master_df[
    (master_df["candidate EDAM term label"].notna()) &
    (master_df["candidate EDAM term label"] != "no_matching_term") &
    (master_df["Validation expert"] == "Yes")
].copy()


In [31]:
validated_df = validated_df.drop(
    columns=[
        "Commentaire expert 1",
        "Commentaire expert 2",
        "Commentaire expert 3",
        "Comment"
    ],
    errors="ignore"
)

In [32]:
validated_df = validated_df[
    [
        "tools",
        "annotation",
        "candidate EDAM term label",
        "Validation expert",
        "hasExactmatch",
        "hasBroader",
        "hasNarrower",
        "EDAM term URI",
        "domain",
        "edam_category",
    ]
]

In [34]:
validated_df

,tools,annotation,candidate EDAM term label,Validation expert,hasExactmatch,hasBroader,hasNarrower,EDAM term URI,domain,edam_category
0,AFNI,anatomical mri,MRI image,Yes,No,Yes,No,http://edamontology.org/data_3442,Neuroimaging,data
1,AFNI,fmri,MRI image,Yes,No,Yes,No,http://edamontology.org/data_3442,Neuroimaging,data
2,AFNI,parametric modulation maps,Statistical estimate score,Yes,No,Yes,No,http://edamontology.org/data_0951,Neuroimaging,data
3,AFNI,nifti,Image format,Yes,No,No,No,http://edamontology.org/format_3549,Neuroimaging,formats
4,AFNI,group statistical analysis,Statistical calculation,Yes,No,No,No,http://edamontology.org/operation_2238,Neuroimaging,operations
...,...,...,...,...,...,...,...,...,...,...
821,mOTUs,metagenomics,Metagenomics,Yes,Yes,No,No,http://edamontology.org/topic_3174,Metagenomic,topics
822,mOTUs,phylogeny,Phylogeny,Yes,Yes,No,No,http://edamontology.org/topic_0084,Metagenomic,topics
823,mOTUs,sequence analysis,Sequence analysis,Yes,Yes,No,No,http://edamontology.org/topic_0080,Metagenomic,topics
824,mOTUs,sequencing,Sequencing,Yes,Yes,No,No,http://edamontology.org/topic_3168,Metagenomic,topics


In [35]:
validated_df.to_csv(
    "EDAM_terms_URI_GT_validated.tsv",
    sep="\t",
    index=False
)

In [2]:

df = pd.read_csv(
    "EDAM_terms_URI_ground_truth_validated.tsv",
    sep="\t"
)

df.head()

,EDAM term URI,candidate EDAM term label,tools,annotation,domain,edam_category
0,http://edamontology.org/data_3707,Biodiversity data,MetaBAT2,abundance profiling,Metagenomic,data
1,http://edamontology.org/data_3707,Biodiversity data,"Kraken2, mOTUs",abundance table,Metagenomic,data
2,http://edamontology.org/data_0925,Sequence Assembly,"Anvio, CONCOCT, MEGAHIT, MetaBAT2, checkM2",contigs,Metagenomic,data
3,http://edamontology.org/operation_3672,Gene functional annotation,Anvio,functional annotations,Metagenomic,data
4,http://edamontology.org/data_2968,Image,Anvio,image,Metagenomic,data


In [5]:
df["tools"] = df["tools"].str.split(r"\s*,\s*")

df = df.explode("tools").reset_index(drop=True)

df

,EDAM term URI,candidate EDAM term label,tools,annotation,domain,edam_category
0,http://edamontology.org/data_3707,Biodiversity data,MetaBAT2,abundance profiling,Metagenomic,data
1,http://edamontology.org/data_3707,Biodiversity data,Kraken2,abundance table,Metagenomic,data
2,http://edamontology.org/data_3707,Biodiversity data,mOTUs,abundance table,Metagenomic,data
3,http://edamontology.org/data_0925,Sequence Assembly,Anvio,contigs,Metagenomic,data
4,http://edamontology.org/data_0925,Sequence Assembly,CONCOCT,contigs,Metagenomic,data
...,...,...,...,...,...,...
621,http://edamontology.org/operation_3443,Image analysis,CellPose,segmentation 2d,Microscopy,operations
622,http://edamontology.org/operation_3443,Image analysis,CellPose,segmentation 3d,Microscopy,operations
623,http://edamontology.org/topic_2229,Cell biology,CellPose,cell segmentation,Microscopy,topics
624,http://edamontology.org/topic_3382,Imaging,EcClem,correlative microscopy,Microscopy,topics


In [6]:
cols = ["tools"] + [c for c in df.columns if c != "tools"]
df = df[cols]

In [7]:
df = df.sort_values(
    by=["tools", "annotation"],
    ignore_index=True
)

In [8]:
df

,tools,EDAM term URI,candidate EDAM term label,annotation,domain,edam_category
0,AFNI,http://edamontology.org/data_3442,MRI image,anatomical mri,Neuroimaging,data
1,AFNI,http://edamontology.org/topic_3384,Medical imaging,brain imaging,Neuroimaging,topics
2,AFNI,http://edamontology.org/data_3442,MRI image,fmri,Neuroimaging,data
3,AFNI,http://edamontology.org/topic_3444,MRI,functional & anatomical MRI preprocesing,Neuroimaging,topics
4,AFNI,http://edamontology.org/operation_2238,Statistical calculation,group statistical analysis,Neuroimaging,operations
...,...,...,...,...,...,...
621,mOTUs,http://edamontology.org/operation_3460,Taxonomic classification,taxonomic annotation,Metagenomic,operations
622,mOTUs,http://edamontology.org/operation_3460,Taxonomic classification,taxonomic profiling,Metagenomic,topics
623,mOTUs,http://edamontology.org/format_3475,TSV,tsv,Metagenomic,formats
624,mOTUs,http://edamontology.org/operation_0337,Visualisation,visualization,Metagenomic,operations


In [9]:
output_file = "GT_EDAM_class_validated_by_tools.tsv"
df.to_csv(
    output_file,
    sep="\t",
    index=False
)
